# Qwen SQL Inference Notebook

This notebook is designed for Google Colab and supports these experiment modes:

- zero-shot + thinking
- zero-shot + non-thinking
- zero-shot + thinking + outlines
- zero-shot + non-thinking + outlines

The output JSON always contains these six fields:

- `instance_id`
- `question`
- `gold_sql`
- `predicted_sql`
- `latency_ms`
- `num_tokens_generated`


In [1]:
!pip install -q transformers accelerate sentencepiece
!pip install -q outlines

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.3/102.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 7.6 MB/s eta 0:00:00


In [2]:
import os
import re
import json
import time
from typing import Dict, Any, List, Optional

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# =========================
# User config
# =========================

INPUT_PATH = "data/wikisql_train_sample50.json"   # Change this to your uploaded JSON path
OUTPUT_DIR = "outputs"

MODEL_NAME = "Qwen/Qwen3-0.6B"

# Options:
# MODE = "thinking" or "non-thinking"
# CONSTRAINT_METHOD = "none" or "outlines"
# TRAIN_MODE = "zero" or "ft"
DATASET_NAME = "wikisql"
MODE = "non-thinking"
CONSTRAINT_METHOD = "none"
TRAIN_MODE = "zero"

MAX_NEW_TOKENS = 128

# For SQL generation, deterministic decoding is usually more stable.
TEMPERATURE = None
TOP_P = None
TOP_K = None
MIN_P = None

TRUST_REMOTE_CODE = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=TRUST_REMOTE_CODE,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=TRUST_REMOTE_CODE,
    torch_dtype="auto",
    device_map="auto",
)

model.eval()
print("Model loaded.")
print("Device:", DEVICE)

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} examples.")
if len(data) > 0:
    print("Sample keys:", list(data[0].keys()))

def get_default_sampling(mode: str) -> Dict[str, Any]:
    """
    Use deterministic decoding for more stable SQL generation.
    """
    return {
        "do_sample": False,
    }


def build_generation_kwargs(
    mode: str,
    max_new_tokens: int,
    temperature: Optional[float],
    top_p: Optional[float],
    top_k: Optional[int],
    min_p: Optional[float],
) -> Dict[str, Any]:
    kwargs = get_default_sampling(mode)
    kwargs["max_new_tokens"] = max_new_tokens

    if kwargs.get("do_sample", False):
        if temperature is not None:
            kwargs["temperature"] = temperature
        if top_p is not None:
            kwargs["top_p"] = top_p
        if top_k is not None:
            kwargs["top_k"] = top_k
        if min_p is not None:
            kwargs["min_p"] = min_p

    return kwargs


def format_schema_text(schema: Dict[str, Any]) -> str:
    """
    Convert schema metadata into a simple text format that is easy for the model to read.
    """
    if not schema:
        return "No schema provided."

    table_names = schema.get("table_names_original") or schema.get("table_names") or []
    column_names = schema.get("column_names_original") or schema.get("column_names") or []
    column_types = schema.get("column_types") or ["unknown"] * len(column_names)

    lines = []

    for table_id, table_name in enumerate(table_names):
        lines.append(f"TABLE: {table_name}")
        lines.append("COLUMNS:")
        for idx, pair in enumerate(column_names):
            if not isinstance(pair, (list, tuple)) or len(pair) != 2:
                continue
            tid, col_name = pair
            if tid != table_id:
                continue
            if col_name == "*":
                continue

            col_type = column_types[idx] if idx < len(column_types) else "unknown"
            lines.append(f"- {col_name} [{col_type}]")
        lines.append("")

    text = "\n".join(lines).strip()
    return text if text else "No schema provided."


def build_plain_prompt(example: Dict[str, Any]) -> str:
    """
    Build a stronger prompt for text-to-SQL generation.
    """
    instance_id = example.get("instance_id", "")
    question = example.get("question", "")
    schema_text = format_schema_text(example.get("schema", {}))

    prompt = f"""You are a text-to-SQL system.

Task:
Convert the question into exactly one SQL query for the given schema.

Rules:
1. Output exactly one SQL query and nothing else.
2. The query must start with SELECT.
3. Use only the table name: table
4. Use only columns that appear in the schema.
5. Preserve string values exactly from the question when possible.
6. Put text values in single quotes.
7. Do not invent columns, values, or conditions.
8. Do not add explanation.
9. Prefer the simplest correct WikiSQL-style query.

Instance ID: {instance_id}

Schema:
{schema_text}

Question:
{question}

SQL:
"""
    return prompt


def build_qwen_prompt(tokenizer, example: Dict[str, Any], mode: str) -> str:
    """
    Build a chat-formatted prompt for Qwen.
    """
    user_prompt = build_plain_prompt(example)

    messages = [
        {"role": "system", "content": "You are a precise text-to-SQL assistant."},
        {"role": "user", "content": user_prompt},
    ]

    enable_thinking = (mode == "thinking")

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )
    return prompt


# def clean_sql(text: str, mode: str = "non-thinking") -> str:
#     """
#     Extract final SQL from model output.

#     In thinking mode, keep only the text after </think>.
#     Then remove markdown fences and keep the SQL starting from SELECT.
#     """
#     if text is None:
#         return ""

#     text = text.strip()

#     # Handle thinking mode
#     if mode == "thinking":
#         if "</think>" in text:
#             text = text.split("</think>", 1)[1].strip()
#         elif "select" in text.lower():
#             text = text[text.lower().rfind("select"):].strip()

#     # Remove markdown code fences
#     text = text.replace("```sql", "").replace("```", "").strip()

#     # Normalize whitespace
#     text = text.replace("\n", " ")
#     text = " ".join(text.split())

#     # Keep only from the first SELECT
#     lower_text = text.lower()
#     if "select" in lower_text:
#         text = text[lower_text.find("select"):].strip()
#     else:
#         return ""

#     if not text.lower().startswith("select"):
#         return ""

#     if text and not text.endswith(";"):
#         text += ";"

#     return text
# def clean_sql(text: str, mode: str = "non-thinking") -> str:
#     """
#     Extract clean SQL from model output.

#     - In thinking mode: keep only text after </think>
#     - Remove special tokens like <|im_end|>
#     - Remove markdown fences
#     - Keep only SQL starting from SELECT
#     - Ensure single trailing semicolon
#     """

#     if text is None:
#         return ""

#     text = text.strip()

#     # --------------------------------------------------
#     # Step 1: Handle thinking mode
#     # --------------------------------------------------
#     if mode == "thinking":
#         if "</think>" in text:
#             text = text.split("</think>", 1)[1].strip()
#         else:
#             # fallback: take last SELECT
#             lower = text.lower()
#             if "select" in lower:
#                 text = text[lower.rfind("select"):].strip()

#     # --------------------------------------------------
#     # Step 2: Remove special tokens (VERY IMPORTANT)
#     # --------------------------------------------------
#     # Remove Qwen / chat template artifacts
#     text = re.sub(r"<\|.*?\|>", "", text)

#     # --------------------------------------------------
#     # Step 3: Remove markdown code blocks
#     # --------------------------------------------------
#     text = text.replace("```sql", "").replace("```", "").strip()

#     # --------------------------------------------------
#     # Step 4: Normalize whitespace
#     # --------------------------------------------------
#     text = text.replace("\n", " ")
#     text = " ".join(text.split())

#     # --------------------------------------------------
#     # Step 5: Keep only SQL starting from SELECT
#     # --------------------------------------------------
#     lower_text = text.lower()
#     if "select" in lower_text:
#         text = text[lower_text.find("select"):].strip()
#     else:
#         return ""

#     # --------------------------------------------------
#     # Step 6: Remove trailing garbage after semicolon
#     # --------------------------------------------------
#     if ";" in text:
#         # text = text.split(";", 1)[0] + ";"
#         text = text.split(";", 1)[0]

#     # --------------------------------------------------
#     # Step 7: Final safety
#     # --------------------------------------------------
#     if not text.lower().startswith("select"):
#         return ""

#     return text
def clean_sql(text: str, mode: str = "non-thinking") -> str:
    if text is None:
        return ""

    text = text.strip()

    if mode == "thinking":
        if "</think>" in text:
            text = text.split("</think>", 1)[1].strip()
        elif "select" in text.lower():
            text = text[text.lower().rfind("select"):].strip()

    text = re.sub(r"<\|.*?\|>", "", text)
    text = text.replace("```sql", "").replace("```", "").strip()
    text = text.replace("\n", " ")
    text = " ".join(text.split())

    lower_text = text.lower()
    if "select" not in lower_text:
        return ""

    text = text[lower_text.find("select"):].strip()

    if ";" in text:
        text = text.split(";", 1)[0].strip()

    # Reject obviously incomplete SQL
    bad_suffixes = ("where", "and", "or", "(", "=", ">", "<", ">=", "<=", "!=")
    if text.lower().endswith(bad_suffixes):
        return ""
    if text.count("(") != text.count(")"):
        return ""

    if not text.lower().startswith("select"):
        return ""

    return text


def validate_output_record(record: Dict[str, Any]) -> None:
    """
    Ensure required keys are included in the output record.
    """
    required_keys = [
        "instance_id",
        "question",
        "gold_sql",
        "predicted_sql",
        "latency_ms",
        "num_tokens_generated",
    ]

    missing = [k for k in required_keys if k not in record]
    if missing:
        raise ValueError(f"Missing required keys: {missing}")


def generate_plain(
    model,
    tokenizer,
    prompt: str,
    generation_kwargs: Dict[str, Any],
):
    """
    Plain Hugging Face generation without constrained decoding.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    start_time = time.perf_counter()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **generation_kwargs,
        )

    latency_ms = (time.perf_counter() - start_time) * 1000.0

    generated_ids = outputs[0][input_len:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=False)
    num_tokens_generated = int(generated_ids.shape[0])

    return text, num_tokens_generated, latency_ms


def build_simple_sql_regex() -> str:
    """
    A minimal regex constraint.
    This is intentionally simple because overly strict regex often hurts performance.
    """
    return r"SELECT\s+.+?\s+FROM\s+table(?:\s+WHERE\s+.+?)?\s*;?"

def build_better_wikisql_regex() -> str:
    # Regex upgraded using the teammate grammar idea while staying in outlines.
    ident = r"[A-Za-z_][A-Za-z0-9_ ]*"
    number = r"-?\d+(?:\.\d+)?"
    quoted = r"'[^']*'"
    bare = r"[A-Za-z0-9_./()\-]+"
    value = rf"(?:{quoted}|{number}|{bare})"
    op = r"(?:=|!=|<|>|<=|>=|LIKE)"
    agg = rf"(?:COUNT\(\*\)|COUNT\({ident}\)|MAX\({ident}\)|MIN\({ident}\)|SUM\({ident}\)|AVG\({ident}\))"
    select_expr = rf"(?:\*|{ident}|{agg})"
    condition = rf"{ident}\s+{op}\s+{value}"

    where_opt = rf"(?:\s+WHERE\s+{condition}(?:\s+(?:AND|OR)\s+{condition})*)?"
    order_opt = rf"(?:\s+ORDER\s+BY\s+{ident}(?:\s+(?:ASC|DESC))?)?"
    limit_opt = rf"(?:\s+LIMIT\s+\d+)?"

    return rf"SELECT\s+(?:DISTINCT\s+)?{select_expr}\s+FROM\s+table{where_opt}{order_opt}{limit_opt}\s*;?"

class OutlinesRunner:
    """
    A version-compatible Outlines wrapper.
    """
    def __init__(self, model, tokenizer):
        import outlines
        from outlines.types import Regex

        self.outlines = outlines
        self.Regex = Regex
        self.tokenizer = tokenizer
        self.outlines_model = outlines.from_transformers(model, tokenizer)

    def generate_regex(
        self,
        prompt: str,
        regex_pattern: str,
        max_new_tokens: int = 128,
    ):
        start_time = time.perf_counter()

        output = self.outlines_model(
            prompt,
            self.Regex(regex_pattern),
            max_new_tokens=max_new_tokens,
        )

        latency_ms = (time.perf_counter() - start_time) * 1000.0

        text = output if isinstance(output, str) else str(output)
        num_tokens_generated = len(
            self.tokenizer.encode(text, add_special_tokens=False)
        )

        return text, num_tokens_generated, latency_ms


def build_output_filename(
    dataset_name: str,
    mode: str,
    train_mode: str,
    constraint_method: str,
) -> str:
    """
    Required naming convention:
    result_{dataset name}_{thinking mode}_{zero/ft}_{constrained decoding methods}.json
    """
    return f"result_{dataset_name}_{mode}_{train_mode}_{constraint_method}.json"



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded.
Device: cuda
Loaded 50 examples.
Sample keys: ['instance_id', 'db', 'question', 'schema', 'gold_sql_query']


In [3]:
# Choose from: "thinking", "non-thinking"
MODE = "non-thinking"
# MODE = "thinking"

# Choose from: "zero", "ft"
TRAIN_MODE = "zero"

# Choose from: "none", "outlines"
# CONSTRAINT_METHOD = "none"
CONSTRAINT_METHOD = "outlines"

MAX_NEW_TOKENS = 100

os.makedirs(OUTPUT_DIR, exist_ok=True)

generation_kwargs = build_generation_kwargs(
    mode=MODE,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    min_p=MIN_P,
)

outlines_runner = None
if CONSTRAINT_METHOD == "outlines":
    outlines_runner = OutlinesRunner(model, tokenizer)

results = []

for idx, example in enumerate(data, start=1):
    prompt = build_qwen_prompt(tokenizer, example, MODE)

    if CONSTRAINT_METHOD == "outlines":
        regex_pattern = build_better_wikisql_regex()
        raw_text, num_tokens_generated, latency_ms = outlines_runner.generate_regex(
            prompt=prompt,
            regex_pattern=regex_pattern,
            max_new_tokens=MAX_NEW_TOKENS,
        )
    else:
        raw_text, num_tokens_generated, latency_ms = generate_plain(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            generation_kwargs=generation_kwargs,
        )

    predicted_sql = clean_sql(raw_text, mode=MODE)

    # Keep all original fields from the input example.
    result = dict(example)

    # Map gold_sql_query -> gold_sql
    gold_sql_value = example.get("gold_sql_query", "")

    result.update({
        "instance_id": example.get("instance_id", ""),
        "question": example.get("question", ""),
        "gold_sql": gold_sql_value,
        "predicted_sql": predicted_sql,
        "latency_ms": round(float(latency_ms), 3),
        "num_tokens_generated": int(num_tokens_generated),
    })

    # Remove the old field if it exists
    if "gold_sql_query" in result:
        del result["gold_sql_query"]

    validate_output_record(result)
    results.append(result)

    if idx <= 3:
        print("=" * 80)
        print(f"Example {idx}")
        print("Question:", result["question"])
        print("Gold SQL:", result["gold_sql"])
        print("Predicted SQL:", result["predicted_sql"])
        print("Latency (ms):", result["latency_ms"])
        print("Generated tokens:", result["num_tokens_generated"])
    # if idx == 3:
    #   break

    if idx % 5 == 0 or idx == len(data):
        print(f"Done {idx}/{len(data)}")

output_filename = build_output_filename(
    dataset_name=DATASET_NAME,
    mode=MODE,
    train_mode=TRAIN_MODE,
    constraint_method=CONSTRAINT_METHOD,
)

output_path = os.path.join(OUTPUT_DIR, output_filename)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved to:", output_path)

Example 1
Question: What is the title of the episode directed by Peter Markle and written by Jerry Stahl?
Gold SQL: SELECT Title FROM table WHERE Directed by = 'Peter Markle' AND Written by = 'Jerry Stahl'
Predicted SQL: SELECT Title FROM table WHERE Directed by = 'Peter Markle' AND Written by = 'Jerry Stahl'
Latency (ms): 33833.024
Generated tokens: 22
Example 2
Question: What's the total number of episodes with production code 2T6705?
Gold SQL: SELECT COUNT(Title) FROM table WHERE Production code = '2T6705'
Predicted SQL: SELECT SERIES COUNT FROM table WHERE Production code = '2T6705'
Latency (ms): 8997.696
Generated tokens: 17
Example 3
Question: How many times was the GFR 2006 equal to 53.2?
Gold SQL: SELECT COUNT(Whites as % of Pop.) FROM table WHERE GFR 2006 = '53.2'
Predicted SQL: SELECT COUNT(*) FROM table WHERE GFR 2006 = '53.2'
Latency (ms): 9331.314
Generated tokens: 20
Done 5/50
Done 10/50
Done 15/50
Done 20/50
Done 25/50
Done 30/50
Done 35/50
Done 40/50
Done 45/50
Done 50/

In [ ]:
# Choose from: "thinking", "non-thinking"
# MODE = "non-thinking"
MODE = "thinking"

# Choose from: "zero", "ft"
TRAIN_MODE = "zero"

# Choose from: "none", "outlines"
# CONSTRAINT_METHOD = "none"
CONSTRAINT_METHOD = "outlines"

MAX_NEW_TOKENS = 1000

os.makedirs(OUTPUT_DIR, exist_ok=True)

generation_kwargs = build_generation_kwargs(
    mode=MODE,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    min_p=MIN_P,
)

outlines_runner = None
if CONSTRAINT_METHOD == "outlines":
    outlines_runner = OutlinesRunner(model, tokenizer)

results = []

for idx, example in enumerate(data, start=1):
    prompt = build_qwen_prompt(tokenizer, example, MODE)

    if CONSTRAINT_METHOD == "outlines":
        regex_pattern = build_better_wikisql_regex()
        raw_text, num_tokens_generated, latency_ms = outlines_runner.generate_regex(
            prompt=prompt,
            regex_pattern=regex_pattern,
            max_new_tokens=MAX_NEW_TOKENS,
        )
    else:
        raw_text, num_tokens_generated, latency_ms = generate_plain(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            generation_kwargs=generation_kwargs,
        )

    predicted_sql = clean_sql(raw_text, mode=MODE)

    # Keep all original fields from the input example.
    result = dict(example)

    # Map gold_sql_query -> gold_sql
    gold_sql_value = example.get("gold_sql_query", "")

    result.update({
        "instance_id": example.get("instance_id", ""),
        "question": example.get("question", ""),
        "gold_sql": gold_sql_value,
        "predicted_sql": predicted_sql,
        "latency_ms": round(float(latency_ms), 3),
        "num_tokens_generated": int(num_tokens_generated),
    })

    # Remove the old field if it exists
    if "gold_sql_query" in result:
        del result["gold_sql_query"]

    validate_output_record(result)
    results.append(result)

    if idx <= 3:
        print("=" * 80)
        print(f"Example {idx}")
        print("Question:", result["question"])
        print("Gold SQL:", result["gold_sql"])
        print("Predicted SQL:", result["predicted_sql"])
        print("Latency (ms):", result["latency_ms"])
        print("Generated tokens:", result["num_tokens_generated"])
    # if idx == 3:
    #   break

    if idx % 5 == 0 or idx == len(data):
        print(f"Done {idx}/{len(data)}")

output_filename = build_output_filename(
    dataset_name=DATASET_NAME,
    mode=MODE,
    train_mode=TRAIN_MODE,
    constraint_method=CONSTRAINT_METHOD,
)

output_path = os.path.join(OUTPUT_DIR, output_filename)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved to:", output_path)

Example 1
Question: What is the title of the episode directed by Peter Markle and written by Jerry Stahl?
Gold SQL: SELECT Title FROM table WHERE Directed by = 'Peter Markle' AND Written by = 'Jerry Stahl'
Predicted SQL: SELECT Title FROM table WHERE Directed by = 'Peter Markle' AND Written by = 'Jerry Stahl'
Latency (ms): 9843.154
Generated tokens: 22
Example 2
Question: What's the total number of episodes with production code 2T6705?
Gold SQL: SELECT COUNT(Title) FROM table WHERE Production code = '2T6705'
Predicted SQL: SELECT * FROM table WHERE Production code = '2T6705'
Latency (ms): 9177.359
Generated tokens: 16
Example 3
Question: How many times was the GFR 2006 equal to 53.2?
Gold SQL: SELECT COUNT(Whites as % of Pop.) FROM table WHERE GFR 2006 = '53.2'
Predicted SQL: SELECT COUNT(*) FROM table WHERE GFR 2006 = '53.2'
Latency (ms): 9520.883
Generated tokens: 20
Done 5/50


In [ ]:
# Choose from: "thinking", "non-thinking"
MODE = "non-thinking"
# MODE = "thinking"

# Choose from: "zero", "ft"
TRAIN_MODE = "zero"

# Choose from: "none", "outlines"
CONSTRAINT_METHOD = "none"
# CONSTRAINT_METHOD = "outlines"

MAX_NEW_TOKENS = 100

os.makedirs(OUTPUT_DIR, exist_ok=True)

generation_kwargs = build_generation_kwargs(
    mode=MODE,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    min_p=MIN_P,
)

outlines_runner = None
if CONSTRAINT_METHOD == "outlines":
    outlines_runner = OutlinesRunner(model, tokenizer)

results = []

for idx, example in enumerate(data, start=1):
    prompt = build_qwen_prompt(tokenizer, example, MODE)

    if CONSTRAINT_METHOD == "outlines":
        regex_pattern = build_better_wikisql_regex()
        raw_text, num_tokens_generated, latency_ms = outlines_runner.generate_regex(
            prompt=prompt,
            regex_pattern=regex_pattern,
            max_new_tokens=MAX_NEW_TOKENS,
        )
    else:
        raw_text, num_tokens_generated, latency_ms = generate_plain(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            generation_kwargs=generation_kwargs,
        )

    predicted_sql = clean_sql(raw_text, mode=MODE)

    # Keep all original fields from the input example.
    result = dict(example)

    # Map gold_sql_query -> gold_sql
    gold_sql_value = example.get("gold_sql_query", "")

    result.update({
        "instance_id": example.get("instance_id", ""),
        "question": example.get("question", ""),
        "gold_sql": gold_sql_value,
        "predicted_sql": predicted_sql,
        "latency_ms": round(float(latency_ms), 3),
        "num_tokens_generated": int(num_tokens_generated),
    })

    # Remove the old field if it exists
    if "gold_sql_query" in result:
        del result["gold_sql_query"]

    validate_output_record(result)
    results.append(result)

    if idx <= 3:
        print("=" * 80)
        print(f"Example {idx}")
        print("Question:", result["question"])
        print("Gold SQL:", result["gold_sql"])
        print("Predicted SQL:", result["predicted_sql"])
        print("Latency (ms):", result["latency_ms"])
        print("Generated tokens:", result["num_tokens_generated"])
    # if idx == 3:
    #   break

    if idx % 5 == 0 or idx == len(data):
        print(f"Done {idx}/{len(data)}")

output_filename = build_output_filename(
    dataset_name=DATASET_NAME,
    mode=MODE,
    train_mode=TRAIN_MODE,
    constraint_method=CONSTRAINT_METHOD,
)

output_path = os.path.join(OUTPUT_DIR, output_filename)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved to:", output_path)

In [ ]:
# Choose from: "thinking", "non-thinking"
# MODE = "non-thinking"
MODE = "thinking"

# Choose from: "zero", "ft"
TRAIN_MODE = "zero"

# Choose from: "none", "outlines"
CONSTRAINT_METHOD = "none"
# CONSTRAINT_METHOD = "outlines"

MAX_NEW_TOKENS = 1000

os.makedirs(OUTPUT_DIR, exist_ok=True)

generation_kwargs = build_generation_kwargs(
    mode=MODE,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    min_p=MIN_P,
)

outlines_runner = None
if CONSTRAINT_METHOD == "outlines":
    outlines_runner = OutlinesRunner(model, tokenizer)

results = []

for idx, example in enumerate(data, start=1):
    prompt = build_qwen_prompt(tokenizer, example, MODE)

    if CONSTRAINT_METHOD == "outlines":
        regex_pattern = build_better_wikisql_regex()
        raw_text, num_tokens_generated, latency_ms = outlines_runner.generate_regex(
            prompt=prompt,
            regex_pattern=regex_pattern,
            max_new_tokens=MAX_NEW_TOKENS,
        )
    else:
        raw_text, num_tokens_generated, latency_ms = generate_plain(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            generation_kwargs=generation_kwargs,
        )

    predicted_sql = clean_sql(raw_text, mode=MODE)

    # Keep all original fields from the input example.
    result = dict(example)

    # Map gold_sql_query -> gold_sql
    gold_sql_value = example.get("gold_sql_query", "")

    result.update({
        "instance_id": example.get("instance_id", ""),
        "question": example.get("question", ""),
        "gold_sql": gold_sql_value,
        "predicted_sql": predicted_sql,
        "latency_ms": round(float(latency_ms), 3),
        "num_tokens_generated": int(num_tokens_generated),
    })

    # Remove the old field if it exists
    if "gold_sql_query" in result:
        del result["gold_sql_query"]

    validate_output_record(result)
    results.append(result)

    if idx <= 3:
        print("=" * 80)
        print(f"Example {idx}")
        print("Question:", result["question"])
        print("Gold SQL:", result["gold_sql"])
        print("Predicted SQL:", result["predicted_sql"])
        print("Latency (ms):", result["latency_ms"])
        print("Generated tokens:", result["num_tokens_generated"])
    # if idx == 3:
    #   break

    if idx % 5 == 0 or idx == len(data):
        print(f"Done {idx}/{len(data)}")

output_filename = build_output_filename(
    dataset_name=DATASET_NAME,
    mode=MODE,
    train_mode=TRAIN_MODE,
    constraint_method=CONSTRAINT_METHOD,
)

output_path = os.path.join(OUTPUT_DIR, output_filename)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved to:", output_path)